In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

df = pd.read_csv("train.csv")
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [19]:
df = df.drop(columns = ["PassengerId", "Name", "Ticket", "Cabin"])
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df = df.drop(columns=["SibSp", "Parch"])
df["AgeBin"] = pd.cut(df["Age"], bins=[0,12,18,30,50,65,100], labels=[0,1,2,3,4,5])
df["AgeBin"] = df["AgeBin"].astype(int)
print(df.isnull().sum())
print(df.head())

Survived      0
Pclass        0
Sex           0
Age           0
Fare          0
Embarked_Q    0
Embarked_S    0
FamilySize    0
AgeBin        0
dtype: int64
   Survived  Pclass  Sex   Age     Fare  Embarked_Q  Embarked_S  FamilySize  \
0         0       3    0  22.0   7.2500       False        True           2   
1         1       1    1  38.0  71.2833       False       False           2   
2         1       3    1  26.0   7.9250       False        True           1   
3         1       1    1  35.0  53.1000       False        True           2   
4         0       3    0  35.0   8.0500       False        True           1   

   AgeBin  
0       2  
1       3  
2       2  
3       3  
4       3  


In [20]:
X = df.drop(columns=["Survived"])
y = df["Survived"]
print(X)
print(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

     Pclass  Sex   Age     Fare  Embarked_Q  Embarked_S  FamilySize  AgeBin
0         3    0  22.0   7.2500       False        True           2       2
1         1    1  38.0  71.2833       False       False           2       3
2         3    1  26.0   7.9250       False        True           1       2
3         1    1  35.0  53.1000       False        True           2       3
4         3    0  35.0   8.0500       False        True           1       3
..      ...  ...   ...      ...         ...         ...         ...     ...
886       2    0  27.0  13.0000       False        True           1       2
887       1    1  19.0  30.0000       False        True           1       2
888       3    1  28.0  23.4500       False        True           4       2
889       1    0  26.0  30.0000       False       False           1       2
890       3    0  32.0   7.7500        True       False           1       3

[891 rows x 8 columns]
0      0
1      1
2      1
3      1
4      0
      ..
886    0
8

In [21]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)
y_predict = model.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_predict))

Accuracy:  0.7597765363128491


In [22]:
for feature, coef in zip(X.columns, model.coef_[0]):
    print(f"{feature}: {coef: .3f}")

Pclass: -1.030
Sex:  2.851
Age: -0.050
Fare:  0.001
Embarked_Q: -0.221
Embarked_S: -0.532
FamilySize: -0.205
AgeBin:  0.102


In [23]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=4,
    min_samples_split=10,
    random_state=7
    )

rf.fit(X_train, y_train)
y_predict = rf.predict(X_test)
print("Test Accuracy: ", accuracy_score(y_test, y_predict))
print("Train Accuracy: ", accuracy_score(y_train, rf.predict(X_train)))

Test Accuracy:  0.7821229050279329
Train Accuracy:  0.8651685393258427


In [24]:
for feature, importance in zip(X.columns, rf.feature_importances_):
    print(f"{feature}: {importance: .3f}")

Pclass:  0.111
Sex:  0.482
Age:  0.094
Fare:  0.150
Embarked_Q:  0.009
Embarked_S:  0.028
FamilySize:  0.083
AgeBin:  0.044


In [25]:
scores = cross_val_score(rf, X, y, cv=5)
print("Accuracy across different test sets: ", scores)
print("Standard Diviation: ", scores.std())
print("Average Accuracy: ", scores.mean())

Accuracy across different test sets:  [0.77094972 0.83146067 0.83146067 0.79775281 0.87078652]
Standard Diviation:  0.033887076616292984
Average Accuracy:  0.8204820789655389
